In [1]:
import pandas as pd
import json
import os
import glob
from pathlib import Path

In [ ]:
#合并LOB 和 trade_week

# ==========================================
# 1. 基础路径配置
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
OUTPUT_DIR = BASE_DIR / 'processed_features'

# 确保总输出文件夹存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 定义需要处理的文件夹列表
WEEKS_TO_PROCESS = ['week1', 'week2']

# ==========================================
# 2. 核心处理函数 (处理单个 Contract)
# ==========================================
# 修改点：将 token_mapping 和 contract_meta 作为参数传入，避免多周循环时作用域混乱
def process_single_contract(jsonl_path, slug, token_mapping, contract_meta):
    print(f"  -> Processing: {slug}")
    trades_list = []
    books_list = []
    
    # 逐行读取以控制内存
    with open(jsonl_path, 'r') as f:
        for line in f:
            # 忽略空行
            if not line.strip(): continue
            
            data = json.loads(line)
            # 兼容数据格式：有些行是一个列表，有些是字典
            records = data if isinstance(data, list) else [data]
            
            for rec in records:
                # 提取时间戳，转为 datetime
                if 'timestamp' not in rec: continue
                ts = pd.to_datetime(int(rec['timestamp']), unit='ms')
                
                # A. 处理交易 (Price Changes)
                if 'price_changes' in rec:
                    for pc in rec['price_changes']:
                        asset_id = str(pc['asset_id'])
                        token_type = token_mapping.get(asset_id, 'Unknown') # 'Up' 或 'Down'
                        trades_list.append({
                            'timestamp': ts,
                            'type': token_type,
                            'price': float(pc['price']),
                            'size': float(pc['size']),
                            'side': pc['side'] # BUY / SELL
                        })
                        
                # B. 处理订单簿 (Book 快照) - 提取买卖五档及总挂单量
                if 'bids' in rec and 'asks' in rec:
                    asset_id = str(rec.get('asset_id', ''))
                    token_type = token_mapping.get(asset_id, 'Unknown')
                    
                    book_entry = {'timestamp': ts, 'type': token_type}
                    
                    # 提取买盘前五档 (Bids)
                    bid_vol_5 = 0
                    bids_list = rec.get('bids', [])
                    for i in range(5):
                        if i < len(bids_list):
                            book_entry[f'bid{i+1}_price'] = float(bids_list[i]['price'])
                            sz = float(bids_list[i]['size'])
                            book_entry[f'bid{i+1}_size'] = sz
                            bid_vol_5 += sz
                        else:
                            book_entry[f'bid{i+1}_price'] = None
                            book_entry[f'bid{i+1}_size'] = 0
                            
                    # 提取卖盘前五档 (Asks)
                    ask_vol_5 = 0
                    asks_list = rec.get('asks', [])
                    for i in range(5):
                        if i < len(asks_list):
                            book_entry[f'ask{i+1}_price'] = float(asks_list[i]['price'])
                            sz = float(asks_list[i]['size'])
                            book_entry[f'ask{i+1}_size'] = sz
                            ask_vol_5 += sz
                        else:
                            book_entry[f'ask{i+1}_price'] = None
                            book_entry[f'ask{i+1}_size'] = 0
                    
                    # 记录前五档汇总 Volume
                    book_entry['bid_vol_5'] = bid_vol_5
                    book_entry['ask_vol_5'] = ask_vol_5
                    
                    books_list.append(book_entry)

    # 转为 DataFrame 进行重采样
    df_trades = pd.DataFrame(trades_list)
    df_books = pd.DataFrame(books_list)
    
    if df_trades.empty or df_books.empty:
        print(f"     [!] Skipping {slug} due to lack of complete trades/books data.")
        return None

    df_trades.set_index('timestamp', inplace=True)
    df_books.set_index('timestamp', inplace=True)

    # ---------------------------------------------------------
    # 分离 Up 和 Down，并按 1秒 (1s) Resample
    # ---------------------------------------------------------
    resampled_frames = []
    
    for token_type in ['Up', 'Down']:
        # 1. 处理 trades (聚合成 OHLCV 格式)
        td = df_trades[df_trades['type'] == token_type]
        if not td.empty:
            td_resampled = td.resample('1s').agg(
                open=('price', 'first'),
                high=('price', 'max'),
                low=('price', 'min'),
                close=('price', 'last'),
                volume=('size', 'sum')
            ).add_prefix(f'{token_type}_trade_')
        else:
            td_resampled = pd.DataFrame()
            
        # 2. 处理 books (取每秒最后一个快照)
        bd = df_books[df_books['type'] == token_type]
        if not bd.empty:
            # 取该秒的最终挂单状态
            bd_resampled = bd.resample('1s').last() 
            bd_resampled = bd_resampled.drop(columns=['type']).add_prefix(f'{token_type}_book_')
        else:
            bd_resampled = pd.DataFrame()
            
        # 合并该方向 (Up 或 Down) 的 trade 和 book
        combined_type = pd.concat([td_resampled, bd_resampled], axis=1)
        resampled_frames.append(combined_type)

    # 3. 横向拼接 Up 和 Down 成为宽表 (Wide Format)
    df_market = pd.concat(resampled_frames, axis=1)
    
    # 向前填充(ffill)快照空缺，因为没有新快照意味着订单簿维持原样
    # 注意：交易(Trades)的OHLC和Volume为空代表这一秒没交易，对于Volume应填0，对价格应ffill
    trade_vol_cols = [col for col in df_market.columns if 'volume' in col]
    df_market[trade_vol_cols] = df_market[trade_vol_cols].fillna(0)
    df_market.ffill(inplace=True) 
    
    # 添加宏观目标特征 (Strike Price 和 Label)
    meta_info = contract_meta.get(slug, {})
    df_market['strike_price'] = meta_info.get('strike_price', None)
    df_market['target_logic'] = 1 if meta_info.get('result_logic') == 'yes' else 0 # 假设 yes=1 代表 Up 赢
    df_market['slug'] = slug
    
    # 过滤掉全为空的行（发生在市场极早期还没有完整数据时）
    df_market.dropna(how='all', inplace=True)
    
    return df_market

# ==========================================
# 3. 外层循环：动态遍历所有的 Week 目录
# ==========================================
for week in WEEKS_TO_PROCESS:
    print(f"\n{'='*50}")
    print(f"🚀 Starting Data Engineering for {week.upper()}")
    print(f"{'='*50}")
    
    # 动态构建该周的路径
    WEEK_DIR = BASE_DIR / week
    CONTRACTS_DIR = WEEK_DIR / 'contracts' / 'btc'
    TRADES_CSV_PATH = WEEK_DIR / f'trades_{week}.csv'
    
    # 为每一周建立单独的输出子文件夹，保持整洁
    WEEK_OUTPUT_DIR = OUTPUT_DIR / week
    WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------
    # A. 构建该周专属的元数据字典
    # ---------------------------------------------------------
    print(f"[1] Loading {TRADES_CSV_PATH.name} to build Metadata Mapping...")
    if not TRADES_CSV_PATH.exists():
        print(f"⚠️ Warning: {TRADES_CSV_PATH} not found, skipping {week}.")
        continue
        
    # 读取 CSV 并确保 token_id 为字符串格式（防止科学计数法精度丢失）
    df_meta = pd.read_csv(TRADES_CSV_PATH, dtype={'token_id': str})
    
    # Token ID -> Up/Down 映射
    token_mapping = df_meta.drop_duplicates('token_id').set_index('token_id')['token_name'].to_dict()
    
    # Slug -> Strike Price & Label 映射
    contract_meta = df_meta.groupby('slug').agg({
        'strike_price': 'first',
        'result_logic': 'first' 
    }).to_dict('index')

    # ---------------------------------------------------------
    # B. 处理该周下的所有 Contracts
    # ---------------------------------------------------------
    jsonl_files = glob.glob(str(CONTRACTS_DIR / '*.jsonl'))
    print(f"[2] Found {len(jsonl_files)} contracts to process in {week}.\n")
    
    for file_path in jsonl_files:
        slug = os.path.basename(file_path).replace('.jsonl', '')
        
        # 调用核心函数，传入该周专属的 mapping
        df_contract_features = process_single_contract(file_path, slug, token_mapping, contract_meta)
        
        # 保存生成的特征表
        if df_contract_features is not None and not df_contract_features.empty:
            output_file = WEEK_OUTPUT_DIR / f"{slug}_features.parquet"
            # 存为 parquet 压缩格式
            df_contract_features.to_parquet(output_file)
            print(f"     [√] Saved -> {output_file.name} (Shape: {df_contract_features.shape})")

print("\n🎉 All weeks processed successfully!")


🚀 Starting Data Engineering for WEEK1
[1] Loading trades_week1.csv to build Metadata Mapping...
[2] Found 627 contracts to process in week1.

  -> Processing: btc-updown-15m-1765153800
     [√] Saved -> btc-updown-15m-1765153800_features.parquet (Shape: (903, 57))
  -> Processing: btc-updown-15m-1764838800
     [√] Saved -> btc-updown-15m-1764838800_features.parquet (Shape: (868, 57))
  -> Processing: btc-updown-15m-1764656100
     [√] Saved -> btc-updown-15m-1764656100_features.parquet (Shape: (859, 57))
  -> Processing: btc-updown-15m-1765081800
     [√] Saved -> btc-updown-15m-1765081800_features.parquet (Shape: (898, 57))
  -> Processing: btc-updown-15m-1764792900
     [√] Saved -> btc-updown-15m-1764792900_features.parquet (Shape: (881, 57))
  -> Processing: btc-updown-15m-1764834300
     [√] Saved -> btc-updown-15m-1764834300_features.parquet (Shape: (878, 57))
  -> Processing: btc-updown-15m-1765107900
     [√] Saved -> btc-updown-15m-1765107900_features.parquet (Shape: (882, 5

In [6]:
#合并 chainlink 和 LOB 

# ==========================================
# 1. 基础路径配置
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
FEATURES_DIR = BASE_DIR / 'processed_features'  # 上一步生成的 parquet 文件夹
WEEKS = ['week1', 'week2']

# ==========================================
# 2. 解析 Chainlink 数据的函数
# ==========================================
def load_and_resample_chainlink(chainlink_dir):
    """提取 Chainlink 中的 btc/usd 价格，并重采样为 1秒"""
    print(f"  -> Loading Chainlink data from {chainlink_dir.name}...")
    jsonl_files = glob.glob(str(chainlink_dir / '*.jsonl'))
    
    records = []
    for file_path in jsonl_files:
        with open(file_path, 'r') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                
                # Chainlink 的数据全在 payload 里
                payload = data.get('payload', {})
                # 我们只需要比特币 (btc/usd) 的价格
                if payload.get('symbol') == 'btc/usd':
                    records.append({
                        # 毫秒时间戳转 datetime
                        'timestamp': pd.to_datetime(int(payload['timestamp']), unit='ms'),
                        'chainlink_price': float(payload['value'])
                    })
                    
    if not records:
        print("     [!] No BTC Chainlink data found!")
        return pd.DataFrame()

    df_link = pd.DataFrame(records)
    df_link.set_index('timestamp', inplace=True)
    
    # 按照 1 秒重采样，并取每一秒的最后一次更新价格
    # ffill() 的意思是：如果这一秒 Chainlink 没发新价格，就沿用上一秒的价格
    df_link = df_link.resample('1s').last().ffill()
    
    # 重置索引，把 timestamp 变回普通列，为 merge_asof 做准备
    df_link = df_link.reset_index().sort_values('timestamp')
    return df_link

# ==========================================
# 3. 整合主循环 (生成每周的 Master Table)
# ==========================================
master_dfs = []

for week in WEEKS:
    print(f"\n{'='*50}")
    print(f"🚀 Building Master Table for {week.upper()}")
    print(f"{'='*50}")
    
    # 路径
    WEEK_CHAINLINK_DIR = BASE_DIR / week / 'chainlink'
    WEEK_FEATURES_DIR = FEATURES_DIR / week
    
    # 1. 获取该周的 Chainlink 1秒价格表
    df_chainlink = load_and_resample_chainlink(WEEK_CHAINLINK_DIR)
    
    # 2. 读取该周所有生成好的 LOB parquet 文件
    parquet_files = glob.glob(str(WEEK_FEATURES_DIR / '*.parquet'))
    print(f"  -> Loading {len(parquet_files)} LOB contract files...")
    
    weekly_lob_dfs = []
    for p_file in parquet_files:
        df = pd.read_parquet(p_file)
        # 上一步我们把 timestamp 设置成了 index，这里我们要把它弹出来变回普通列
        df = df.reset_index() 
        weekly_lob_dfs.append(df)
        
    # 将几百个 contract 纵向拼成一个超级大表
    df_weekly_lob = pd.concat(weekly_lob_dfs, ignore_index=True)
    
    # merge_asof 强制要求时间戳列必须是升序排列 (Sorted)
    df_weekly_lob = df_weekly_lob.sort_values('timestamp')
    
    # 3. 见证奇迹的时刻：Merge As-Of
    print("  -> Merging LOB features with Chainlink Oracle prices...")
    df_merged = pd.merge_asof(
        left=df_weekly_lob,          # 左表：高频的订单簿特征
        right=df_chainlink,          # 右表：Chainlink 价格
        on='timestamp',              # 按照时间戳对齐
        direction='backward'         # 向后寻找（只使用过去最近的价格，绝不透支未来）
    )
    
    # 保存该周的 Master Table
    output_path = FEATURES_DIR / f"{week}_master.parquet"
    df_merged.to_parquet(output_path)
    print(f"  [√] {week} Master Table saved! Shape: {df_merged.shape}")
    
    master_dfs.append(df_merged)

# ==========================================
# 4. 生成终极全量表 (Two-Week Full Dataset)
# ==========================================
print(f"\n{'='*50}")
print("🌌 Concatenating into Final Two-Week Master Dataset...")
df_final = pd.concat(master_dfs, ignore_index=True)
df_final = df_final.sort_values(['slug', 'timestamp']) # 最终按照合约和时间排序

final_path = FEATURES_DIR / "FINAL_MASTER_DATASET.parquet"
df_final.to_parquet(final_path)

print(f"🎉 All done! Your Final ML-ready Dataset is saved at:")
print(f"   {final_path.name} (Total Shape: {df_final.shape})")


🚀 Building Master Table for WEEK1
  -> Loading Chainlink data from chainlink...
  -> Loading 627 LOB contract files...
  -> Merging LOB features with Chainlink Oracle prices...
  [√] week1 Master Table saved! Shape: (539388, 59)

🚀 Building Master Table for WEEK2
  -> Loading Chainlink data from chainlink...
  -> Loading 608 LOB contract files...
  -> Merging LOB features with Chainlink Oracle prices...
  [√] week2 Master Table saved! Shape: (520726, 59)

🌌 Concatenating into Final Two-Week Master Dataset...
🎉 All done! Your Final ML-ready Dataset is saved at:
   FINAL_MASTER_DATASET.parquet (Total Shape: (1060114, 59))
